In [0]:
# Gold Layer: Seller / Dark Store Performance Scorecard

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, avg, sum as spark_sum, when, round as spark_round,
    current_timestamp, max as spark_max, unix_timestamp, coalesce
)

spark = SparkSession.builder.getOrCreate()

GOLD_TABLE = "ecomstore.ecomlake.gold_seller_performance"

order_items = spark.read.table("ecomstore.ecomlake.silver_order_items")
orders      = spark.read.table("ecomstore.ecomlake.silver_orders")
shipments   = spark.read.table("ecomstore.ecomlake.silver_shipments")
returns     = spark.read.table("ecomstore.ecomlake.silver_returns")
sellers_scd = spark.read.table("ecomstore.ecomlake.silver_sellers_scd").filter(col("is_current") == True)

# 1. PRE-AGGREGATE SHIPMENTS
order_shipments = (
    shipments
    .groupBy("order_id")
    .agg(
        spark_max("promised_delivery_date").alias("promised_delivery_date"),
        spark_max("actual_delivery_date").alias("actual_delivery_date")
    )
    # Calculate exact fulfillment time in minutes for high-velocity tracking
    .withColumn("fulfillment_time_minutes", 
        (unix_timestamp(col("actual_delivery_date")) - unix_timestamp(col("promised_delivery_date"))) / 60
    )
)

# 2. BUILD SELLER METRICS
seller_metrics = (
    order_items.alias("oi")
    .join(orders.select("order_id", "status").alias("o"), on="order_id", how="left")
    .join(order_shipments.alias("s"), on="order_id", how="left")
    .join(returns.select("order_item_id", "return_id").alias("r"), on="order_item_id", how="left")
    
    .groupBy("oi.seller_id")
    .agg(
        count("oi.order_item_id").alias("total_items_sold"),
        spark_sum("oi.total_price").alias("total_gmv"),
        
        # Operational Counts
        count(when(col("o.status") == "delivered", True)).alias("delivered_count"),
        count(when(col("o.status") == "cancelled", True)).alias("cancelled_count"),
        count(when(col("r.return_id").isNotNull(), True)).alias("return_count"),
        
        # Ensure the order is actually delivered before counting it as on-time
        count(when(
            (col("o.status") == "delivered") & 
            (col("s.actual_delivery_date") <= col("s.promised_delivery_date")), True
        )).alias("on_time_deliveries"),
        
        avg("s.fulfillment_time_minutes").alias("avg_fulfillment_time_minutes")
    )
    # 3. SAFE MATH (Prevent Division by Zero)
    .withColumn("fulfillment_rate_pct",
        when(col("total_items_sold") > 0, spark_round(col("delivered_count") / col("total_items_sold") * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("return_rate_pct",
        when(col("total_items_sold") > 0, spark_round(col("return_count") / col("total_items_sold") * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("on_time_delivery_pct",
        when(col("delivered_count") > 0, spark_round(col("on_time_deliveries") / col("delivered_count") * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("avg_fulfillment_time_minutes", spark_round(col("avg_fulfillment_time_minutes"), 2))
    .withColumn("_gold_processed_at", current_timestamp())
)

# 4. ENRICH & WRITE
final_seller_perf = (
    seller_metrics.alias("sm")
    .join(
        sellers_scd.select("seller_id", "seller_name", "seller_tier", "city").alias("sd"),
        on="seller_id",
        how="left"
    )
)

(
    final_seller_perf
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact", "true")
    .saveAsTable(GOLD_TABLE)
)